# Statistical-ML Connections

## Learning Objectives
1. Derive OLS as MLE under Gaussian noise and verify the equivalence numerically
2. Show Ridge as MAP with Gaussian prior and Lasso as MAP with Laplace prior
3. Implement the EM algorithm for Gaussian mixtures from scratch
4. Use bootstrap resampling to quantify model uncertainty and compare OLS vs Ridge vs Lasso

In [ ]:
# Cell 2: Imports and reproducibility
import numpy as np
import scipy.stats as stats
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import cross_val_score

np.random.seed(42)

plt.rcParams['figure.figsize'] = (12, 5)
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.3

print('Libraries loaded.')

## Level 1: OLS as MLE under Gaussian Noise

Derive that minimizing MSE is identical to maximizing the log-likelihood under N(0, sigma^2) residuals. Verify numerically that OLS coefficients = MLE coefficients.

In [ ]:
# Cell 4: OLS = MLE under Gaussian noise
# Statistical model: y_i = w^T x_i + eps_i, eps_i ~ N(0, sigma^2)
# Log-likelihood: sum_i log p(y_i|x_i, w, sigma) = sum_i log N(y_i; w^T x_i, sigma^2)
#   = -n/2 * log(2*pi*sigma^2) - 1/(2*sigma^2) * sum_i (y_i - w^T x_i)^2
# Maximizing over w: equivalent to minimizing sum_i (y_i - w^T x_i)^2 = MSE (up to constant)

rng = np.random.default_rng(42)
N, P = 100, 4
SIGMA = 1.5  # Known noise standard deviation

# True parameters
w_true = np.array([2.0, -1.5, 0.8, 3.0])

# Generate data
X = rng.normal(0, 1, (N, P))
y = X @ w_true + rng.normal(0, SIGMA, N)

# --- Method 1: OLS (minimize MSE) ---
# OLS closed form: w_OLS = (X^T X)^{-1} X^T y
w_ols = np.linalg.lstsq(X, y, rcond=None)[0]

# --- Method 2: MLE (maximize log-likelihood) ---
# For Gaussian noise, MLE and OLS give identical w estimates
# We verify by evaluating log-likelihood at the OLS solution and at a perturbed solution
def log_likelihood_gaussian(w: np.ndarray, X: np.ndarray, y: np.ndarray, sigma: float) -> float:
    """Log-likelihood of Gaussian regression model.
    
    log L(w) = -n/2*log(2*pi*sigma^2) - 1/(2*sigma^2) * ||y - Xw||^2
    """
    n = len(y)
    residuals = y - X @ w
    return -n / 2 * np.log(2 * np.pi * sigma**2) - np.sum(residuals**2) / (2 * sigma**2)


def mse(w: np.ndarray, X: np.ndarray, y: np.ndarray) -> float:
    """Mean squared error loss."""
    residuals = y - X @ w
    return np.mean(residuals**2)


# Verify: OLS minimizes MSE and maximizes log-likelihood simultaneously
w_perturbed = w_ols + rng.normal(0, 0.1, P)

print('OLS = MLE Equivalence Verification:')
print(f'  True w:              {w_true}')
print(f'  OLS/MLE w:           {w_ols.round(4)}')
print(f'  Perturbed w:         {w_perturbed.round(4)}')
print()
print(f'  MSE at OLS w:        {mse(w_ols, X, y):.6f}')
print(f'  MSE at perturbed w:  {mse(w_perturbed, X, y):.6f}  <- higher')
print()
print(f'  Log-lik at OLS w:    {log_likelihood_gaussian(w_ols, X, y, SIGMA):.4f}')
print(f'  Log-lik at perturb:  {log_likelihood_gaussian(w_perturbed, X, y, SIGMA):.4f}  <- lower')
print()
print('=> Minimizing MSE = maximizing Gaussian log-likelihood (identical solution)')

# MLE estimate of sigma^2: sigma^2_MLE = 1/n * ||y - Xw||^2
residuals_ols = y - X @ w_ols
sigma_mle = np.sqrt(np.mean(residuals_ols**2))
sigma_unbiased = np.sqrt(np.sum(residuals_ols**2) / (N - P))
print(f'  MLE sigma: {sigma_mle:.4f} (biased)')
print(f'  Unbiased sigma: {sigma_unbiased:.4f}')
print(f'  True sigma: {SIGMA:.4f}')

# Plot log-likelihood vs MSE along a 1D slice through coefficient space
alpha_range = np.linspace(-3, 3, 200)
ll_vals = [log_likelihood_gaussian(w_ols + alpha * np.ones(P)/P, X, y, SIGMA)
           for alpha in alpha_range]
mse_vals = [mse(w_ols + alpha * np.ones(P)/P, X, y) for alpha in alpha_range]

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
axes[0].plot(alpha_range, mse_vals, color='navy', lw=2)
axes[0].axvline(0, color='red', ls='--', lw=1.5, label='OLS minimum (alpha=0)')
axes[0].set_xlabel('Perturbation alpha (uniform shift of w)')
axes[0].set_ylabel('MSE')
axes[0].set_title('MSE: Minimized at OLS solution')
axes[0].legend()

axes[1].plot(alpha_range, ll_vals, color='green', lw=2)
axes[1].axvline(0, color='red', ls='--', lw=1.5, label='MLE maximum (alpha=0)')
axes[1].set_xlabel('Perturbation alpha')
axes[1].set_ylabel('Log-likelihood')
axes[1].set_title('Log-lik: Maximized at same OLS solution')
axes[1].legend()

plt.suptitle('OLS = MLE under Gaussian Noise (same minimum/maximum)', fontsize=11, fontweight='bold')
plt.tight_layout()
plt.savefig('ols_mle_equivalence.png', dpi=80, bbox_inches='tight')
plt.show()
print('Level 1 complete.')

## Level 2: Ridge as MAP (Gaussian Prior) and Bias-Variance Tradeoff

Derive Ridge regression as MAP estimation with a Gaussian prior on weights: -log posterior = MSE + lambda*||w||^2. Vary lambda to trace the bias-variance tradeoff.

In [ ]:
# Cell 6: Ridge = MAP with Gaussian prior; bias-variance tradeoff
# MAP objective: max p(w|data) = max p(data|w) * p(w)
# With Gaussian prior p(w) = N(0, tau^2 I):
#   log p(w|data) = log-likelihood + log p(w)
#   = -(1/2sigma^2)*||y-Xw||^2 - (1/2tau^2)*||w||^2 + const
#   Equivalently: minimize ||y-Xw||^2 + lambda*||w||^2 where lambda = sigma^2/tau^2
# This IS Ridge regression.

rng = np.random.default_rng(7)
N_BV = 80
P_BV = 20  # More features than are truly relevant
N_TEST = 2000  # Large test set for unbiased error estimate

# True model uses only first 5 features
w_true_bv = np.zeros(P_BV)
w_true_bv[:5] = np.array([2.0, -1.5, 1.0, 0.8, -0.5])

# Generate train and test data
X_train_bv = rng.normal(0, 1, (N_BV, P_BV))
X_test_bv = rng.normal(0, 1, (N_TEST, P_BV))
y_train_bv = X_train_bv @ w_true_bv + rng.normal(0, 1.0, N_BV)
y_test_bv = X_test_bv @ w_true_bv + rng.normal(0, 1.0, N_TEST)

# Sweep lambda (Ridge alpha) on log scale
lambdas = np.logspace(-3, 3, 60)
train_errors = []
test_errors = []
coef_norms = []

for lam in lambdas:
    ridge = Ridge(alpha=lam, fit_intercept=False)
    ridge.fit(X_train_bv, y_train_bv)
    train_pred = ridge.predict(X_train_bv)
    test_pred = ridge.predict(X_test_bv)
    train_errors.append(np.mean((y_train_bv - train_pred)**2))
    test_errors.append(np.mean((y_test_bv - test_pred)**2))
    coef_norms.append(np.linalg.norm(ridge.coef_))

# OLS (lambda=0) and Ridge MAP
ols_bv = LinearRegression(fit_intercept=False).fit(X_train_bv, y_train_bv)
ols_test_err = np.mean((y_test_bv - ols_bv.predict(X_test_bv))**2)
best_lambda = lambdas[np.argmin(test_errors)]
print(f'Bias-Variance Tradeoff: N={N_BV}, P={P_BV}, true sparse features=5')
print(f'OLS test MSE: {ols_test_err:.4f}')
print(f'Best Ridge lambda: {best_lambda:.4f}, test MSE: {min(test_errors):.4f}')
print(f'Improvement over OLS: {(ols_test_err - min(test_errors))/ols_test_err*100:.1f}%')

# MAP interpretation: prior variance tau^2 = sigma^2 / lambda
sigma2_est = np.mean((y_train_bv - ols_bv.predict(X_train_bv))**2)
tau2_best = sigma2_est / best_lambda
print(f'\nMAP interpretation: lambda={best_lambda:.4f} = sigma^2/tau^2')
print(f'Estimated sigma^2 = {sigma2_est:.4f}')
print(f'Implied prior variance tau^2 = {tau2_best:.4f}')

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Bias-variance curve
axes[0].semilogx(lambdas, train_errors, 'b-', lw=2, label='Train MSE (decreasing with lambda)')
axes[0].semilogx(lambdas, test_errors, 'r-', lw=2, label='Test MSE (U-shaped)')
axes[0].axvline(best_lambda, color='black', ls='--', lw=1.5, label=f'Best lambda={best_lambda:.3f}')
axes[0].set_xlabel('Lambda (regularization strength)')
axes[0].set_ylabel('MSE')
axes[0].set_title('Bias-Variance Tradeoff via Ridge (MAP)')
axes[0].legend()

# Coefficient shrinkage with increasing lambda
axes[1].semilogx(lambdas, coef_norms, color='navy', lw=2)
axes[1].axvline(best_lambda, color='red', ls='--', lw=1.5, label='Best lambda')
axes[1].set_xlabel('Lambda')
axes[1].set_ylabel('||w||_2 (coefficient norm)')
axes[1].set_title('Shrinkage: Larger Lambda = Stronger Prior')
axes[1].legend()

plt.suptitle('Ridge = MAP (Gaussian Prior): Bias-Variance Tradeoff', fontsize=11, fontweight='bold')
plt.tight_layout()
plt.savefig('ridge_map_bv.png', dpi=80, bbox_inches='tight')
plt.show()
print('Level 2 complete.')

## Real-World Example 1: Lasso as Laplace MAP — Sparsity and Proximal Gradient

Lasso corresponds to a Laplace prior. Implement coordinate descent (soft-thresholding) for L1 and show that sparse features are exactly zeroed.

In [ ]:
# Cell 8: Lasso = MAP with Laplace prior; sparsity via soft-thresholding
# Laplace prior: p(w_j) ∝ exp(-lambda*|w_j|)
# MAP objective: minimize ||y - Xw||^2 + lambda*||w||_1
# Solution: coordinate descent with soft-thresholding
#   w_j <- sign(rho_j) * max(|rho_j| - lambda/2, 0)
#   where rho_j = X_j^T (y - X_{-j} w_{-j}) / ||X_j||^2

def lasso_coordinate_descent(X: np.ndarray, y: np.ndarray, lam: float,
                               max_iter: int = 1000, tol: float = 1e-6) -> np.ndarray:
    """Lasso via coordinate descent (soft-thresholding).
    
    Iterates over each weight j, computing the optimal w_j given all others.
    Soft-thresholding operator S(z, t) = sign(z) * max(|z| - t, 0)
    sets small weights exactly to zero — this gives Lasso's sparsity.
    """
    n, p = X.shape
    w = np.zeros(p)
    # Precompute squared column norms for efficiency
    col_norms_sq = np.sum(X**2, axis=0)

    for iteration in range(max_iter):
        w_old = w.copy()
        for j in range(p):
            # Partial residual: y - X @ w + X_j * w_j
            r_j = y - X @ w + X[:, j] * w[j]
            # Univariate OLS coefficient (unconstrained)
            rho_j = X[:, j] @ r_j / col_norms_sq[j]
            # Soft-thresholding: this is the Lasso solution for coordinate j
            threshold = lam / (2 * col_norms_sq[j])
            w[j] = np.sign(rho_j) * max(abs(rho_j) - threshold, 0)

        # Check for convergence
        if np.max(np.abs(w - w_old)) < tol:
            break

    return w


rng = np.random.default_rng(33)
N_L, P_L = 120, 15
# True model: only 4 nonzero features
w_true_l = np.zeros(P_L)
w_true_l[[0, 2, 7, 11]] = [3.0, -2.0, 1.5, -1.0]

X_l = rng.normal(0, 1, (N_L, P_L))
y_l = X_l @ w_true_l + rng.normal(0, 0.8, N_L)

# Lasso regularization path (sweep lambda)
lam_path = np.logspace(-2, 1.5, 50)
w_path_custom = np.array([lasso_coordinate_descent(X_l, y_l, lam=lam) for lam in lam_path])

# sklearn Lasso for verification
from sklearn.linear_model import Lasso as SkLasso
w_sk_best = SkLasso(alpha=1.0, fit_intercept=False, max_iter=5000).fit(X_l, y_l).coef_

print(f'Lasso (lambda=1.0): custom vs sklearn comparison')
w_custom_1 = lasso_coordinate_descent(X_l, y_l, lam=2.0)  # 2*alpha for sklearn equiv
print(f'  {'Feature':<10}  {'True':>8}  {'Custom':>10}  {'Sklearn':>10}')
for j in range(P_L):
    nonzero_flag = ' *' if abs(w_true_l[j]) > 0 else ''
    print(f'  {j:<10}  {w_true_l[j]:>8.3f}  {w_custom_1[j]:>10.4f}  {w_sk_best[j]:>10.4f}{nonzero_flag}')

n_zeros_custom = np.sum(np.abs(w_custom_1) < 1e-6)
print(f'\nNon-zero features (custom Lasso): {P_L - n_zeros_custom} / {P_L}')
print(f'Non-zero features (sklearn Lasso): {np.sum(np.abs(w_sk_best) > 1e-6)} / {P_L}')

# Regularization path plot
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

for j in range(P_L):
    color = 'red' if abs(w_true_l[j]) > 0 else 'steelblue'
    lw = 2 if abs(w_true_l[j]) > 0 else 0.8
    alpha = 0.9 if abs(w_true_l[j]) > 0 else 0.4
    axes[0].semilogx(lam_path, w_path_custom[:, j], color=color, lw=lw, alpha=alpha)

axes[0].axhline(0, color='black', lw=0.5)
axes[0].set_xlabel('Lambda (Laplace prior scale)')
axes[0].set_ylabel('Coefficient')
axes[0].set_title('Lasso Path: Red = truly nonzero features')

# Sparsity count
sparsity = np.sum(np.abs(w_path_custom) < 1e-6, axis=1)
axes[1].semilogx(lam_path, sparsity, color='navy', lw=2)
axes[1].axhline(P_L - 4, color='red', ls='--', lw=1.5,
                label=f'True sparsity: {P_L-4} zeros')
axes[1].set_xlabel('Lambda')
axes[1].set_ylabel('Number of zero coefficients')
axes[1].set_title('Sparsity vs Lambda')
axes[1].legend()

plt.suptitle('Lasso = MAP (Laplace Prior): Sparsity via Soft-Thresholding', fontsize=11, fontweight='bold')
plt.tight_layout()
plt.savefig('lasso_laplace_map.png', dpi=80, bbox_inches='tight')
plt.show()

## Real-World Example 2: EM Algorithm for Gaussian Mixture Models

Implement the EM algorithm for a 2-component Gaussian mixture. E-step computes soft assignments (responsibilities); M-step updates means, variances, and mixing weights.

In [ ]:
# Cell 10: EM for Gaussian Mixture Model (GMM)
# Model: p(x) = sum_k pi_k * N(x; mu_k, sigma_k^2)
# EM iterates:
#   E-step: r_{ik} = pi_k * N(x_i; mu_k, sigma_k^2) / sum_j pi_j N(x_i; mu_j, sigma_j^2)
#   M-step: N_k = sum_i r_{ik};
#           pi_k = N_k / N;
#           mu_k = (1/N_k) sum_i r_{ik} x_i;
#           sigma_k^2 = (1/N_k) sum_i r_{ik} (x_i - mu_k)^2

def em_gmm_1d(x: np.ndarray, k: int = 2, max_iter: int = 100,
              tol: float = 1e-6, rng=None) -> dict:
    """EM algorithm for 1D Gaussian Mixture Model.
    
    Alternates between computing soft assignments (E-step) and
    maximizing expected complete-data log-likelihood (M-step).
    
    Returns fitted parameters and log-likelihood history.
    """
    if rng is None:
        rng = np.random.default_rng(0)
    n = len(x)
    # Random initialization of means
    mu = rng.choice(x, k, replace=False)
    sigma = np.full(k, x.std())
    pi = np.ones(k) / k  # Equal mixing weights
    log_liks = []

    for iteration in range(max_iter):
        # --- E-step: compute responsibilities r_{ik} ---
        # r_{ik} = pi_k * N(x_i | mu_k, sigma_k) / normalization
        log_resp = np.array([
            np.log(pi[comp] + 1e-300) + stats.norm.logpdf(x, mu[comp], sigma[comp])
            for comp in range(k)
        ]).T  # shape: (n, k)
        # Numerically stable softmax normalization
        log_resp -= log_resp.max(axis=1, keepdims=True)
        resp = np.exp(log_resp)  # Unnormalized
        resp /= resp.sum(axis=1, keepdims=True)  # Normalize

        # Compute log-likelihood for convergence check
        log_lik = np.sum(np.log(
            sum(pi[comp] * stats.norm.pdf(x, mu[comp], sigma[comp]) for comp in range(k))
            + 1e-300
        ))
        log_liks.append(log_lik)

        # --- M-step: update parameters ---
        N_k = resp.sum(axis=0)  # Effective count per component
        mu_new = (resp * x[:, None]).sum(axis=0) / N_k
        sigma_new = np.sqrt(
            (resp * (x[:, None] - mu_new)**2).sum(axis=0) / N_k
        )
        pi_new = N_k / n

        # Check convergence (change in log-likelihood)
        if iteration > 0 and abs(log_liks[-1] - log_liks[-2]) < tol:
            break

        mu, sigma, pi = mu_new, sigma_new, pi_new

    return {'mu': mu, 'sigma': sigma, 'pi': pi, 'log_liks': log_liks,
            'responsibilities': resp, 'n_iter': iteration + 1}


rng = np.random.default_rng(5)
# Generate data from a known 2-component mixture
TRUE_PI = [0.4, 0.6]
TRUE_MU = [-2.5, 2.5]
TRUE_SIGMA = [0.8, 1.2]

n_gmm = 500
assignments = rng.choice(2, n_gmm, p=TRUE_PI)
x_gmm = np.array([rng.normal(TRUE_MU[a], TRUE_SIGMA[a]) for a in assignments])

gmm_result = em_gmm_1d(x_gmm, k=2, max_iter=200, rng=rng)

print(f'EM for Gaussian Mixture Model (N={n_gmm}, K=2):')
print(f'Converged in {gmm_result["n_iter"]} iterations')
print(f'\n{'Parameter':<12}  {'True':>10}  {'EM Estimate':>13}')
print('-' * 38)
for k_idx in range(2):
    print(f'pi_{k_idx}        {TRUE_PI[k_idx]:>10.4f}  {gmm_result["pi"][k_idx]:>13.4f}')
    print(f'mu_{k_idx}        {TRUE_MU[k_idx]:>10.4f}  {gmm_result["mu"][k_idx]:>13.4f}')
    print(f'sigma_{k_idx}     {TRUE_SIGMA[k_idx]:>10.4f}  {gmm_result["sigma"][k_idx]:>13.4f}')

## Real-World Example 3: Bootstrap Uncertainty + OLS vs Ridge vs Lasso Comparison

Use bootstrap to quantify coefficient uncertainty. Then run a comprehensive comparison of OLS, Ridge, and Lasso on a realistic regression problem.

In [ ]:
# Cell 12: Bootstrap uncertainty + comprehensive OLS/Ridge/Lasso comparison

# ---- Part A: Bootstrap for coefficient uncertainty ----
rng_boot = np.random.default_rng(77)
N_BOOT_DATA = 80
P_BOOT = 5

w_true_boot = np.array([2.0, -1.5, 0.8, 0.0, 0.0])  # Last 2 are irrelevant
X_boot = rng_boot.normal(0, 1, (N_BOOT_DATA, P_BOOT))
y_boot = X_boot @ w_true_boot + rng_boot.normal(0, 1.0, N_BOOT_DATA)

N_BOOT = 1000
ols_boot_coefs = []
ridge_boot_coefs = []

for _ in range(N_BOOT):
    idx = rng_boot.choice(N_BOOT_DATA, N_BOOT_DATA, replace=True)
    ols_b = LinearRegression(fit_intercept=False).fit(X_boot[idx], y_boot[idx])
    ridge_b = Ridge(alpha=5.0, fit_intercept=False).fit(X_boot[idx], y_boot[idx])
    ols_boot_coefs.append(ols_b.coef_)
    ridge_boot_coefs.append(ridge_b.coef_)

ols_boot = np.array(ols_boot_coefs)
ridge_boot = np.array(ridge_boot_coefs)

print('Bootstrap Coefficient Uncertainty (1000 resamples, N=80, P=5):')
print(f'{'Feature':<10}  {'True':>8}  {'OLS mean':>10}  {'OLS 95% CI':>20}  {'Ridge mean':>12}  {'Ridge 95% CI':>20}')
print('-' * 90)
for j in range(P_BOOT):
    ols_ci = (np.percentile(ols_boot[:, j], 2.5), np.percentile(ols_boot[:, j], 97.5))
    ridge_ci = (np.percentile(ridge_boot[:, j], 2.5), np.percentile(ridge_boot[:, j], 97.5))
    print(f'{j:<10}  {w_true_boot[j]:>8.3f}  '
          f'{ols_boot[:, j].mean():>10.4f}  '
          f'[{ols_ci[0]:>6.3f}, {ols_ci[1]:>6.3f}]  '
          f'{ridge_boot[:, j].mean():>12.4f}  '
          f'[{ridge_ci[0]:>6.3f}, {ridge_ci[1]:>6.3f}]')

# ---- Part B: Comprehensive comparison OLS vs Ridge vs Lasso ----
rng_cmp = np.random.default_rng(99)
N_CMP, P_CMP = 100, 20
N_TEST_CMP = 5000

w_sparse = np.zeros(P_CMP)
w_sparse[[0, 3, 7, 12]] = [3.0, -2.0, 1.5, -1.0]  # Only 4 nonzero

X_cmp = rng_cmp.normal(0, 1, (N_CMP, P_CMP))
X_test_cmp = rng_cmp.normal(0, 1, (N_TEST_CMP, P_CMP))
y_cmp = X_cmp @ w_sparse + rng_cmp.normal(0, 1.0, N_CMP)
y_test_cmp = X_test_cmp @ w_sparse + rng_cmp.normal(0, 1.0, N_TEST_CMP)

models_cmp = [
    ('OLS', LinearRegression(fit_intercept=False)),
    ('Ridge(alpha=1)', Ridge(alpha=1.0, fit_intercept=False)),
    ('Ridge(alpha=10)', Ridge(alpha=10.0, fit_intercept=False)),
    ('Lasso(alpha=0.1)', Lasso(alpha=0.1, fit_intercept=False, max_iter=10000)),
    ('Lasso(alpha=0.5)', Lasso(alpha=0.5, fit_intercept=False, max_iter=10000)),
]

print(f'\n\nComparison: N={N_CMP}, P={P_CMP}, true nonzero features=4')
print(f'{'Method':<20}  {'Test MSE':>10}  {'||w||_1':>8}  {'||w||_2':>8}  {'N nonzero':>10}')
print('-' * 62)

all_coefs = []
for name, model in models_cmp:
    model.fit(X_cmp, y_cmp)
    test_mse = np.mean((y_test_cmp - model.predict(X_test_cmp))**2)
    coef = model.coef_
    all_coefs.append(coef)
    n_nonzero = np.sum(np.abs(coef) > 1e-6)
    print(f'{name:<20}  {test_mse:>10.4f}  {np.sum(np.abs(coef)):>8.3f}  '
          f'{np.linalg.norm(coef):>8.3f}  {n_nonzero:>10}')

# Final comparison plot
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# EM log-likelihood convergence
axes[0].plot(gmm_result['log_liks'], color='navy', lw=2)
axes[0].set_xlabel('EM Iteration')
axes[0].set_ylabel('Log-Likelihood')
axes[0].set_title('EM Convergence for GMM')

# Bootstrap CI width: OLS vs Ridge
ols_ci_widths = [np.percentile(ols_boot[:, j], 97.5) - np.percentile(ols_boot[:, j], 2.5)
                 for j in range(P_BOOT)]
ridge_ci_widths = [np.percentile(ridge_boot[:, j], 97.5) - np.percentile(ridge_boot[:, j], 2.5)
                   for j in range(P_BOOT)]
x_pos = np.arange(P_BOOT)
axes[1].bar(x_pos - 0.2, ols_ci_widths, 0.4, color='firebrick', alpha=0.7, label='OLS 95% CI width')
axes[1].bar(x_pos + 0.2, ridge_ci_widths, 0.4, color='steelblue', alpha=0.7, label='Ridge CI width')
axes[1].set_xlabel('Feature index')
axes[1].set_ylabel('95% Bootstrap CI Width')
axes[1].set_title('Bootstrap Uncertainty: OLS vs Ridge')
axes[1].legend()

# Coefficient comparison
method_names = [n for n, _ in models_cmp]
x_feat = np.arange(P_CMP)
for i, (name, coefs_i) in enumerate(zip(method_names, all_coefs)):
    axes[2].plot(x_feat, coefs_i, 'o-', alpha=0.7, lw=1.2, ms=5, label=name)
axes[2].plot(x_feat, w_sparse, 'k*', ms=10, zorder=10, label='True coefs')
axes[2].axhline(0, color='black', lw=0.5)
axes[2].set_xlabel('Feature index')
axes[2].set_ylabel('Coefficient')
axes[2].set_title('Coefficient Comparison: OLS vs Ridge vs Lasso')
axes[2].legend(fontsize=7)

plt.suptitle('Statistical-ML Connections: EM, Bootstrap, Regularization', fontsize=11, fontweight='bold')
plt.tight_layout()
plt.savefig('stat_ml_comparison.png', dpi=80, bbox_inches='tight')
plt.show()

print('\nKey Takeaways:')
print('- OLS = MLE under Gaussian noise: changing the noise model changes the loss')
print('- Ridge = MAP with Gaussian prior (lambda = sigma^2/tau^2)')
print('- Lasso = MAP with Laplace prior: produces exact zeros via soft-thresholding')
print('- EM maximizes marginal likelihood by alternating E and M steps')
print('- Bootstrap gives principled uncertainty without distributional assumptions')